# Apache Kafka Integration: Producers, Consumers, and Schema Handling (Optional)

**Module**: 09_streaming  
**Notebook**: 06_kafka_integration.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark Structured Streaming integrates with Apache Kafka: reading topics, consumer offsets, parsing key-value payloads, and writing to Kafka sinks.
- **Why does this topic exist?**: Kafka + Spark is the dominant real-time streaming architecture in enterprise tech.
- **What problem does it solve?**: Deserializing binary Kafka payloads and managing consumer offsets.
- **Where is it used in real-world Data Engineering?**: Enterprise event streams, IoT ingestion, and transaction event buses.

---

## 2. Kafka + Spark Integration Architecture

```
Apache Kafka Cluster:
[Topic:'orders-stream', Partitions: 0, 1, 2]
                   |
                   v (1 Task per Kafka Partition!)
Spark Structured Streaming (Driver & Executors):
  - readStream.format("kafka").option("subscribe","orders-stream")
  - Each task connects directly to a Kafka Broker partition
                   |
                   v Deserialization & Business Logic (JSON / Avro)
  - df.select(from_json(col("value").cast("string"), schema))
                   |
                   v Output to Storage or Write Back to Kafka Sink
```

> **Note on Infrastructure**: This notebook provides complete production code examples. In environments without an active Kafka container broker, the code serves as an authoritative architectural reference.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Kafka_Integration_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Production Kafka Consumer Template

```python
# Reading from Kafka
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","kafka:9092")
    .option("subscribe","orders_raw")
    .option("startingOffsets","latest") # or"earliest"
    .option("failOnDataLoss","false")
    .load()
)

# Kafka streams contain fixed schema: [key, value, topic, partition, offset, timestamp, timestampType]
parsed_stream = (
    kafka_stream
    .selectExpr("CAST(key AS STRING) as order_key","CAST(value AS STRING) as payload","timestamp")
    .select(F.col("order_key"), F.from_json(F.col("payload"), orders_schema).alias("order"),"timestamp")
    .select("order_key","order.*","timestamp")
)
```

---

## Experiment: Kafka Producer Specification

- **1. What are we testing?**: Explain how Spark writes to Kafka using key and value columns.
- **2. What do we expect?**: DataFrame with'key'and'value'columns written directly to Kafka topic.

In [ ]:
print("""
Writing to Kafka requires 'topic', 'key', and 'value' columns:
(
    df
    .selectExpr("CAST(order_id AS STRING) as key", "to_json(struct(*)) as value")
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "orders_processed")
    .option("checkpointLocation", "/opt/spark-data/warehouse/chk_kafka_out")
    .start()
)
""")

**3. What happened?**: Displayed Kafka writer structure.  
**4. Why did it happen?**: Kafka expects byte arrays for key and value.  
**5. Practical Engineering Lesson**: Always serialize DataFrame records with to_json() or to_avro() before writing to Kafka.

---

## Practical Exercises

### Exercise 1 (Beginner): Kafka Stream Schema
**Problem Statement**: What columns does readStream.format('kafka') return?  
**Input / Context**: Schema

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): failOnDataLoss Option
**Problem Statement**: Why set failOnDataLoss=false?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Docker Compose Kafka Service
**Problem Statement**: Show how to add Kafka to Docker Compose.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Kafka Stream Schema):
```python
print('key, value, topic, partition, offset, timestamp, timestampType')
```

#### Solution 2 (failOnDataLoss Option):
```python
print('Prevents stream failure if Kafka log retention purges older offsets before Spark finishes reading them.')
```

#### Solution 3 (Docker Compose Kafka Service):
```python
print('Add wurstmeister/kafka or confluentinc/cp-kafka service mapped to spark-net.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Kafka Standard**: 1 Spark task per Kafka topic partition.
- **Serialization**: Values arrive as binary; cast to String and parse with from_json.

### Essential Technical Terms
- `bootstrap.servers`: Kafka broker connection string.
- `startingOffsets`: earliest or latest.

### Common Engineering Mistakes
- **Forgetting to cast binary value**: Attempting string operations directly on binary columns fails.

### Interview Questions on This Topic
- **Q: How is parallelism determined when reading from Kafka in Structured Streaming?**
  - *A*: Each partition of the Kafka topic is read by exactly one Spark task, so parallelism matches the number of Kafka topic partitions.

### What to Study Next
Module 09 complete! Proceed to Module 10: Performance and Optimization.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')